First, import the torch and torchvision libaries to get the Faster R-CNN model

In [40]:
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn

Now, load the model

In [28]:
# Check MPS is available (Apple Silicon GPU)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Running on {device}")

# Load pretrained Faster R-CNN
model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
model.to(device)
model.eval()

Running on mps


FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(

Let's test the code first

In [91]:
from PIL import Image, ImageDraw
import torchvision.transforms as T

img = Image.open('./test3.png').convert("RGB")

# Convert img to tensor (what the model expects)
transform = T.ToTensor()
img_tensor = transform(img).to(device)

model.eval()
# Run inference
with torch.no_grad():
    predictions = model([img_tensor])

# See what it found
print(predictions)


[{'boxes': tensor([[8.3507e+02, 3.3808e+02, 9.0326e+02, 4.1234e+02],
        [9.9452e+02, 1.2171e+03, 1.0886e+03, 1.3118e+03],
        [1.6895e+03, 5.0605e+02, 1.8073e+03, 6.2282e+02],
        [9.8136e+02, 5.5695e+01, 1.0733e+03, 1.6836e+02],
        [8.1560e+02, 2.9563e+02, 8.9852e+02, 4.2753e+02],
        [1.9759e+03, 2.4137e+02, 2.0776e+03, 4.1628e+02],
        [7.4266e+02, 2.5342e+02, 9.3613e+02, 4.4186e+02],
        [2.7444e+02, 6.1303e+02, 3.4059e+02, 8.0486e+02],
        [9.3123e+02, 4.3773e+01, 1.0613e+03, 1.7582e+02],
        [8.8133e+02, 1.1791e+03, 9.7241e+02, 1.2955e+03],
        [8.8403e+02, 1.1663e+03, 9.4274e+02, 1.2323e+03],
        [8.7199e+02, 3.8082e+01, 1.0107e+03, 1.7075e+02],
        [1.1326e+03, 2.5560e+02, 1.2097e+03, 4.3982e+02],
        [1.1856e+03, 3.9940e+00, 1.2885e+03, 1.0588e+02],
        [7.3427e+02, 2.8735e+02, 8.5384e+02, 4.0680e+02],
        [5.5971e+02, 3.2969e+02, 6.8045e+02, 4.7591e+02],
        [4.7089e+02, 7.4600e-01, 6.3485e+02, 8.6934e+01],
   

Function do draw classification boundaries on image

In [88]:
def debug_visualize(image_path, predictions):
    img = Image.open(image_path)
    draw = ImageDraw.Draw(img)
    boxes = predictions[0]["boxes"]
    labels = predictions[0]["labels"]
    scores = predictions[0]["scores"]
    for box, label, score in zip(boxes, labels, scores):
        if score > 0.3:
            box = box.cpu().tolist()
            draw.rectangle(box, outline="green", width=2)

            # draw label text at top left corner of box
            draw.text(
                (box[0], box[1] - 15),  # slightly above the box
                f"{label}",
                fill="green"
            )
    img.show()

Calling the visualization

In [92]:
debug_visualize("./test3.png", predictions)

# Model Finetuning
We will finetune the model in the following steps:
- Take in the finetuned dataset
- Convert to the proper format (tensors)
- Then finetune the model based on the format

# Steps 1 & 2
First we will take in the finetuned dataset and convert it into a list of dictionaries where each element of the dictinoary is either "boxes", i.e. the location of boxes on our classified images, or "labels", i.e. 1 since we are classifying tree (1) and not tree (0).

In [ ]:
import json

with open("./Training Classifier/annotations.json") as f:
    annotations = json.load(f)

# list of (image_path, target) pairs
dataset = []
for image_name, boxes in annotations.items():
    target = {
        "boxes":  torch.tensor(boxes, dtype=torch.float32),
        "labels": torch.ones(len(boxes), dtype=torch.int64)  # all class 1 = tree
    }
    dataset.append(("./Training Classifer/raw_images/" + image_name, target))

# check it looks right
# print(dataset[4])

('./Training Classifer/raw_images/13.jpg', {'boxes': tensor([[ 478.,  674.,  555.,  737.],
        [ 628.,  699.,  699.,  764.],
        [ 618., 1161.,  691., 1234.],
        [ 265., 1159.,  361., 1223.],
        [  14., 1053.,   51., 1122.]]), 'labels': tensor([1, 1, 1, 1, 1])})


In [62]:
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as T
import json
import torch

class TreeDataset(Dataset):
    def __init__(self, annotations_path):
        with open(annotations_path) as f:
            all_annotations = list(json.load(f).items())
        
        self.annotations = [a for a in all_annotations if a[1] != []]
        self.transform = T.ToTensor()

    def __len__(self):
        return len(self.annotations)  # how many images total

    def __getitem__(self, idx):
        image_name, boxes = self.annotations[idx]
        
        # your exact code from before
        img = Image.open("./Training Classifier/raw_images/" + image_name).convert("RGB")
        img_tensor = self.transform(img)
        
        target = {
            "boxes":  torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.ones(len(boxes), dtype=torch.int64),
            "image_id": torch.tensor([idx])
        }
        
        return img_tensor, target

# then use it like
dataset_full = TreeDataset("./Training Classifier/annotations.json")
print(f"Total images: {len(dataset_full)}")

Total images: 25


# Step 3
Now we are modifying the classification layer of our Faster R-CNN model. Instead of multiple classes like airplanes, persons, chairs, etc., we will now have only 2 classes: tree (1) or no tree (0)

In [47]:
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

def get_fine_tune_ready_model():
    # load a model pre-trained on COCO
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")

    num_classes = 2  # 1 class (tree) + background

    # get number of input features for the classifier (from the conv. layers / pooling)
    in_features = model.roi_heads.box_predictor.cls_score.in_features

    # replace the pre-trained head with a new one
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # Freezing most parameters (everything but the final head/FC layer)
    for param in model.backbone.parameters():
        param.requires_grad = False
    for param in model.rpn.parameters():
        param.requires_grad = False
    
    return model

# Step 4
Instead of writing our own training loop and evaluation function, let's use the given functions from pytorch directly. 

In [49]:
import os

base = "https://raw.githubusercontent.com/pytorch/vision/main/references/detection"
files = ["engine.py", "utils.py", "coco_utils.py", "coco_eval.py", "transforms.py"]

for f in files:
    os.system(f"curl -o finetune_helpers/{f} {base}/{f}")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  4063  100  4063    0     0  23149      0 --:--:-- --:--:-- --:--:-- 23217
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  8388  100  8388    0     0  50563      0 --:--:-- --:--:-- --:--:-- 50836
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  8409  100  8409    0     0  49321      0 --:--:-- --:--:-- --:--:-- 49464
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  6447  100  6447    0     0  38650      0 --:--:-- --:--:-- --:--:-- 38604
  % Total    % Received % Xferd  Average Speed   Tim

# Step 5
Now we have all functions necessary to finetune the model.
Let's begin.

In [79]:
import sys
sys.path.append("./finetune_helpers")
from engine import train_one_epoch, evaluate
import utils

# train on the accelerator or on the CPU, if an accelerator is not available
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# our dataset has two classes only - background and tree
num_classes = 2
# use our dataset and defined transformations
dataset = TreeDataset('./Training Classifier/annotations.json')
#dataset_test = TreeDataset('./Training Classifier/annotations.json')

# split the dataset in train and test set
indices = torch.randperm(len(dataset)).tolist()
# dataset = torch.utils.data.Subset(dataset, indices[:-6])
#dataset_test = torch.utils.data.Subset(dataset_test, indices[-6:])

# define training and validation data loaders
data_loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=utils.collate_fn
)

# data_loader_test = torch.utils.data.DataLoader(
#     dataset_test,
#     batch_size=1,
#     shuffle=False,
#     collate_fn=utils.collate_fn
# )

# get the model using our helper function
model = get_fine_tune_ready_model()

# move model to the right device
model.to(device)

# construct an optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(
    params,
    lr=0.005,
    momentum=0.9,
    weight_decay=0.0005
)

# and a learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=3,
    gamma=0.1
)

# let's train it just for 2 epochs
num_epochs = 10

for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=10)
    # update the learning rate
    lr_scheduler.step()
    # evaluate on the test dataset
    # evaluate(model, data_loader_test, device=device)

print("That's it!")

/Users/paulrabel/tree_detector/finetune_helpers/engine.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=scaler is not None):


Epoch: [0]  [ 0/13]  eta: 0:01:20  lr: 0.000421  loss: 4.4603 (4.4603)  loss_classifier: 0.6634 (0.6634)  loss_box_reg: 0.0997 (0.0997)  loss_objectness: 3.3709 (3.3709)  loss_rpn_box_reg: 0.3263 (0.3263)  time: 6.1554  data: 0.0644
Epoch: [0]  [10/13]  eta: 0:00:04  lr: 0.004584  loss: 5.2737 (5.4464)  loss_classifier: 0.5259 (0.5302)  loss_box_reg: 0.1569 (0.1631)  loss_objectness: 4.2732 (4.4995)  loss_rpn_box_reg: 0.2375 (0.2535)  time: 1.5929  data: 0.0389
Epoch: [0]  [12/13]  eta: 0:00:01  lr: 0.005000  loss: 5.2173 (5.0610)  loss_classifier: 0.5224 (0.4718)  loss_box_reg: 0.1543 (0.1442)  loss_objectness: 4.2732 (4.2091)  loss_rpn_box_reg: 0.2375 (0.2360)  time: 1.4214  data: 0.0369
Epoch: [0] Total time: 0:00:18 (1.4216 s / it)
Epoch: [1]  [ 0/13]  eta: 0:00:08  lr: 0.005000  loss: 3.2102 (3.2102)  loss_classifier: 0.2434 (0.2434)  loss_box_reg: 0.1319 (0.1319)  loss_objectness: 2.7135 (2.7135)  loss_rpn_box_reg: 0.1215 (0.1215)  time: 0.6379  data: 0.0452
Epoch: [1]  [10/13]  